# 📋 STR01 · 미션 07 | 결핍 부대 긴급 물자 전환 결정을 위한 군수처장 HITL 승인 워크플로우 설계

**교관용 Solution · 외부 배포 금지**  
CMS 편성: 4시간 · 교육용 가상 데이터 사용

---

## 📖 1. 상황과 이번 미션의 문제

가상의 20개 조직과 5종 물품의 기록을 이용해 **결핍 부대 긴급 물자 전환 결정을 위한 군수처장 HITL 승인 워크플로우 설계** 과제를 수행합니다.

실제 조직의 기록이 아닌 실습자료이며, 코드를 실행한 결과도 실제 현장 상태를 뜻하지 않습니다.

**이번 미션에서 해결할 문제:** 고정 목표 재고 정책과 예측 기반 동적 목표 정책을 같은 50일 자료로 재현해 결핍량, 재고 회전, 비용을 비교한다.

경계 이하 부대에 다음 보급일까지 버틸 물량을 여유 부대에서 옮기거나 보급소 예비로 충당하는 제안을 만들고, 군수처장의 승인·반려와 결정 근거 기재를 거쳐야만 집행하게 한다.

자동 승인, 권한 없는 승인 등 우회 시도를 차단하고 감사 이력을 해시로 검증한다.

**만들 결과물:** `src/optimization/str01_simulator.py`, `reports/str01_policy_report.md`, `outputs/learner_audit.json`, `outputs/M07/results.json`, `forms/STR01_M07_forms.md`

## 📌 2. 처음 시작하는 방법

1. 아래 **환경 준비 셀**을 실행합니다. Colab에서는 메뉴의 **런타임 → 모두 실행** 대신 각 셀의 결과를 먼저 확인하면서 진행해도 됩니다.
2. 핵심 함수의 `TODO`를 AI와 함께 구현합니다. 처음에는 함수의 입력·출력과 왜 필요한지 질문하세요.
3. 첫 실행 결과를 확인하고 실패하면 오류 메시지를 AI에게 설명해 달라고 요청합니다.
4. AI가 제안한 코드 중 최소 3가지를 원자료와 실행 결과로 대조하고 수정 이유를 기록합니다.
5. 활동별 작성 칸을 채운 뒤 **실제 AI 대화 원문**을 보존하고 마지막 저장 셀로 ZIP을 만듭니다.

> **보안과 평가:** 실제 조직·위치·재고·작전 자료, 개인정보를 외부 AI에 입력하지 않습니다.

- 모델 성능은 **실제 출력 로그**로 판단하며, 예시값을 실측으로 적지 않습니다.
- 실행 성공이나 ZIP 저장은 학습 PASS·실제 승인과 다릅니다.
- 폐쇄망에서는 허용된 Python/Jupyter 환경에서 실행하세요.
- 인터넷이 필요한 `pip` 설치는 폐쇄망에서 실패할 수 있으므로 사전 패키지 준비가 필요합니다.

**실습 전 필수:** ① 환경 설정 셀만 실행하면 됩니다.

실습자료(미션 lesson·forms, 공통 안내)가 노트북에 내장되어 있어 ZIP 업로드 없이 자동으로 준비됩니다.

준비 완료 문구를 확인한 뒤 다음 셀로 진행하세요.


In [ ]:
# @title ① 실습자료 자동 준비 및 환경 설정 (ZIP 업로드 불필요) (실행 버튼을 누르세요) { display-mode: "form" }
# 이 셀은 코드 숨김 상태로 배포됩니다. 소스코드를 열지 않아도 실행할 수 있습니다.
from pathlib import Path
import os, sys, json, re, math, time, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 업로드 파일명은 자유이며, .zip 확장자와 유효한 ZIP 내용만 확인합니다.
current_dir = Path.cwd().resolve()
notebook_directory = globals().get('notebook_directory', current_dir)
workspace_name = 'STR01_실습프로젝트_작업공간'
WORKSPACE = next(
    (folder for folder in (*reversed(current_dir.parents), current_dir) if folder.name == workspace_name),
    current_dir / workspace_name,
)
STR01_PROJECT_READY = False

def _prepare_project_zip(raw_bytes):
    global STR01_PROJECT_READY, STR01_PROJECT_DIR
    import io
    if not zipfile.is_zipfile(io.BytesIO(raw_bytes)):
        raise ValueError('유효한 ZIP 파일이 아닙니다. 압축파일 내용을 확인하세요.')
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw_bytes)) as archive:
        for member in archive.infolist():
            name = member.filename.replace('\\', '/')
            target = (WORKSPACE / name).resolve()
            if not target.is_relative_to(WORKSPACE):
                raise ValueError('안전하지 않은 압축 경로가 발견되어 중단했습니다: ' + name)
            # 심볼릭 링크는 작업공간 외부 접근 가능성을 고려해 거부합니다.
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('심볼릭 링크가 포함된 ZIP은 허용하지 않습니다: ' + name)
        for member in archive.infolist():
            target = (WORKSPACE / member.filename.replace('\\', '/')).resolve()
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            if target.is_file():
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open('xb') as destination:
                import shutil
                shutil.copyfileobj(source, destination)
    os.chdir(WORKSPACE)
    STR01_PROJECT_DIR = WORKSPACE
    STR01_PROJECT_READY = True
    print('✅ 실습자료 준비 완료 (기존 파일은 보존):', WORKSPACE)
    print('✅ 이제 다음 교관용 실습 코드셀을 실행할 수 있습니다.')

# 실습자료가 노트북 안에 내장되어 있어 ZIP 업로드 없이 자동으로 준비됩니다.
# 내장 파일 2개: STR01_M07_forms.md, STR01_M07_lesson.md
import base64 as _embedded_b64
_EMBEDDED_ZIP_B64 = (
    'UEsDBBQAAAAIAAAAIVwHKwt0jQkAACcXAAASAAAAU1RSMDFfTTA3X2Zvcm1zLm1k7VhbaxzJFX6fX1FgFhLQzbvrBAQhOCQQg/cl630KQThrBRRsS+jysGEe'
    'WnLLjDSzeOTVSD1yjzKydXVGUWtmJLXwKAH/FO9bV/V/yHfOqeoZaeQLgX1bbOyeruo6t+9855y6ob6+96eRm+qrkV8rs7li/GNl1lZMsabeeasqaUZp5Xul'
    'Tz1d8lQSt5N4SelGx2yWlan7aTWgLaZeMTVfmdBPK6FKTiJTCEwzMJs76o937t1VZvnc1GJlXpTS+Ua6WtJbodmIlPG3k5afy5kXZd2KVfpsKYm8UfXntOLj'
    'gL8oOvZ4UZliqP/d0PWCqbXNYvD2rF8A62pOQlOfT6t4qs+bjQP6UOlznx5v3voMbwtJVFY6KOtl7NlsJK26SjdKsMOE++rzkc9UurZjnsyTLbrVhjjsGLD2'
    '4PPQVC5+gQXygXnVSeJoQJlKAd+700xYN3veL5NWh7VeCPRuwwTbuoUDSx7trbVZu3ohrQRJs6R0FJhgXpcDfQCHbJb1s6oyeyvp2hJvXX8KXeGVoVxOt7yk'
    'eaFwNjQZVdiIg5R+fZCuhMrUOqTjg7G/6p2O+tXQLbNeNn6obg2NmLADf0PQavLfVesP+JAD+/bM2k8Wn/vJSTmtHOhiQRe3h+ACXz+DaXtQdwW6Ler6DsRi'
    'V4MOZH8U1BfJMfREZMRvEuq3Z7BL17fxf6utdwIyFu5JzjrJcUQa6oX95HzeSXIGezA4Ys0CuBG+XTbLJ3ZTLnfjhkOEqJrL5e2Lt2cwXOW7xkUeFvBCrNT/'
    'XHKGQpRH/sozel/gd7RPjs4zel4CbH49fVJjg8K6ykPE4OCgeu+/WIdhIqznL15LfBCJ96wgMldW2EDr7PXDtFK3qSV2krN1s5AGcC5wgpyx7iS7WzGcbmou'
    'GcmchQYboBCq5Cym3DTnyB1zGiJxaENxG4da10NgAzn+Sfb22fOxF2zXRoQAOM0pQmVoCLPKSdNjA5EuABseggKgRIZQFmV6ym986GwG04A8PqQxVm7fcTmV'
    'OaZHz+S0RHwlSeZ8fmmH9aIu+Lp2cWWpEDCubCxMuyD69m7ZXtGtk2tF26hITsOqTgNMAXGHZOsV38EIUE268T25LdRRSNnrmd0avdZbtVxukBUFutNSWRf3'
    'Mx4vghDLo+rd2hYdoosnkKeYwCLCPy/Yt2a9rTdLilOCv8aht++ARZTe20piAGh9UTfapurZ3BglqXXE4IS8Rl5+RfG9WNU/QEEhS+bBF+CUgBwsIaTvHF7b'
    'NtpcNXiVzwf97xGz7eudNkdmoZFWDkcJJmBRIniwHpFfF8Ubq6Z5SLJ0I84Yvu5RgRDkd/c6gz+4t5tR+mAVmnwEZ+/FP4obqwYiZLHMcVZnVsDVmq12D/OK'
    'Fy06nIou5uwn5lAuHeQ7OqDqWy9QGgkwuifmbC742/r1YmazSLGMl+/GsLgNSgBjLpbM8na6HF9HNII0DrClSlv88D1qc73iqL9LraIvviyhWmGb9fVPQjlX'
    'qhIV362o18NsqxJCJWea6D/i5CeEeeQmdKMapPfK3CZU2rp4jrLVRoHBuSjtGTO1fLMQObf1xIfqfE/5ytt4MtgRn+UduLhU0rWOPopTX9zLAec6ZRNJN336'
    'ma1xgl5dc7WsV4Me3P+/le2TiF16Nxhktj0Wa6WBAE5DslraOnjwEFlLNVm2Rs9hwKmP3AKdxVES+R8ge11sgJLwwVGMR/JbmShVnxbo6LxLgJ8ES/1mD7KR'
    '7UIWCPuTMqxdoKZIMtqGgJnShWwxRkLp03lOD2odfKLzs1jvwl5Xq9l3PV8DQKYGtB7F6H9dP+cHRMvS6nA/Z8kVnVHS9ogu0qcgZrDrk1BqALKAmJ8fexmD'
    '30jj9qO3qosEdSQI7EJXWQxdHH70KhzytHJiig3IaaSBn9Ugqsj71IPmbavIvQa1jGm1Qt0/3jf/pRy5CEmoHth2uTcKk2aDKzNa550g2+zg8EEqnpl4NPfw'
    '/uw4XvEfNM5HsdWVJJX2mfv8YyRgF4tciyq+LlYzhiBl/WPUN6sgg/uUmvFLuchr1/XP4qsq2poq9f9mz2fDr5TuvN1CHY34SXTqSQa2usuXNqQMoI+AHis3'
    '8UyjHVgKHcQoF9x4i8VHgXLDDHfRTOsb+469Lk9dDnXWyEAmo97pkHsFlGemt/45kSrfpcFNACfn6r2S7URtryBj4+V5kbXikREyyMMkkWbFjVUGi0WdkAkV'
    '3H24DH5Da4QJEHGjiEtPRLF0QIHZ+NRilci8csGDDNdKO7hkdU/6LCl+b8/QICEom76LqjiZvrYFeK0g49tvLRbz6nMXD0bkqJqZ/nZ4cmp24tHEP+7PTkw+'
    'Hp6ZnR65OWaVm5wemvqO4uKUhQ+kVRHFRF0k7dTkw4lvvxvAGPQbGjnePX3+numZ3Ex9sxgypG7b+WX4d5cnSm4w8KLWuTWCf7Lm4Ev+wcOvzFno1WXsuoKQ'
    '0NbZQL9+o8thNqmh+ABkPRMecEFjKxVfzIannlg2IrVB2r+IosNcL4faRs7CC9WYFPVsLnaPxse0H7yFkJjTA7N1bOGF5JbcdnQkzIvCshIOCxvjE9uXdYMI'
    'Ucj5u+MzM5OPCQzMBkIGJIHVEqYHNG171g+BLxwEGDM/33dce99hg5wls+0U0MWhs5OKcB4gatIJ2DSVOjws5SJjZovIbiAE0YJuC2xCrk0EmwaOE6+5GEHN'
    'ZqQXUZ1ilT4vUGMj8yDtx05zHjpP1zzbI1Eaw9l0jyaXEARcOAH7KI2RR1ls+qsR9dpEYKu24EusHJu7OukudVx0EFqgnlXaT96A7tuLmQ1RzFsErVDt6ztf'
    'fXP39r0//D670KOerYoOlpOs9xKIwk3G73ZP613uY3pLEzJgW6vdEARW3a0hvpdzua8Yy10TAVzmbWZoGF+5sBlN+KWw89yYBuQgfeTJxRfdRNnJNvvWIm+z'
    'oaNIANR340RNP6GaTl9o0/WIu4JD4oI+bQiqMS2ZYl2/jGzxRoUhKJMdWUvnm2ab5Ml0TT7Se0s8Ybx+04Mt6iFfwRc7w9Kv0yADhe3NGQ7jts4Vfw61pb1+'
    'ovnSEg02AuKfVGsG1PT41OT07Ixdk7oyJi+HHj0YUJNzs1NzWH84fn/68fj02P25BxOzQ38HI3bXIHR4enxm7uHsjF352+T0o5lhvmkew+oY/8Z5xKJCm+wW'
    'aaVfLsp0wLO+kHB2fcVOkRmGYMXXhcQm1nQ3IFCrHG/JGIohGwmLF06ECOT2Q1pirqoV9F/pZoESsxj2efN/UEsDBBQAAAAIAAAAIVyB9dMPBwsAAKcdAAAT'
    'AAAAU1RSMDFfTTA3X2xlc3Nvbi5tZO1ZXU8c2RF951dcyVopkdgB/LFJkKKVE1sJiq2s1s5LoghYQyIibCzAK22eBtxYY2YcBnsG9+Ae0rML5mNnd5uhgUEG'
    'ReKnrN/63v4POVV1u6cH8MqPUeQnoD/uvVV16pxTzSV15+7n/QPqdv8v1I/5iopaQVx9pvRBXpfyKmqHUfup0s1js1ZWxnfimkuPGL9q6o4ynhNXPRXtB6bg'
    'mpZr1jbU74fu3lJm8cjU28q8KsVzzbhS0g3PrAbKOOvRntPTc+mSMo/n4tp2tHes9M52vOz19ERBHhfVHz4bwu6OcepKF0NdPMLO9Lrj2UPpPWy81oz2/NND'
    '86Skd7ZOD2UNdeOPd0wtn8QQtU70Xqg3ju16cXVbFwu6uJ5Txi2YdgPhBTpwFZ6iKCmo3QVTd1W8uoWT0Zum6Onvm9ovmHpoFlzjHeMsSh850X45rro4RG93'
    'yszLssIPTssFyTP1fHe6JFOyrt4sKbO5HK88xdJ6sWJz2Z1EPhXnEXtQMJLOV2W9h4QXXLytdKuC0lAq4qWnyGtPz8fJE3HVwUNK/9BGODk6lWmFCHPdLO7j'
    '7EgzbSzrdBKWvo+UPy6o303MqvhfbePsqiu5a8hRFDhv8xVapd2IgudIljJVRxfdt/lqZ29ZdVAha8i1Lbytpc0+AeJMcZZqxp9LH7Y1clQU1CiZ1/pNHa8A'
    'I99Qgmi12EVqQlsW/e+nvckW8WoJ8fZS+czqNi2SFJITSYCJCFz1EPnP1hNx1UsWJtguOiqZzTyy7MTFNtUXm3A7vGwaz++8SJg17vcIJNoN9HwCNICWYsTe'
    'fOADJPnIFtz4nqkW+GSbJf1iXfCVwQuhUyAD0Aeu9tct4Cmh0eExNqLEIWDCCf4ye2VTPe5CVtQqqSRkStxSzS6JrQ5KhFzz8gmfRrpYv9hVAB6yx+2w5DAE'
    'gy1d3JIWQC3KZr5JidP+Bp0e+adHEV7UypvX9QSqH6u/CAD/ait5tsFOD8+TCfOS2fcAg7hGOZpD9ehFW0g1cO0jXC3gFEq7ZT55tuDG21KX+z9S8coGCIaT'
    'uxdiu0xyJfEnP0uY4JtjJBG4QS18J0Wo56PsP2eI4tTzrn7dNO468RHVm+oW8un8gs0yKmTcOV129XaQpprLwI9yltGeuUyLzAfmwAOkBtXM9L2+qYezE/cn'
    '/jk6OzH1oG9mdrp/YHhm4v6jydHZqencw6+k9X0v2tu3LUCdjr/iJ3RhmRoU59Lz4enhb6cmR7+gRieCSC5K3yu90I4X2/pgLrkSVwibxq/jsvrz0Gd8wpOK'
    'fuENqpH3ONhIDhDYN8UmfmwhvwzGfepM7ODnUVqLHNP6NjlT/MpR14cIzuaAsoIq5dGRXSSknwXUnU4wqLqOODI2Ojva9/nN6zdu38zdHxth0t86xlYA5QkC'
    'sksgy9j7V1HgEREiHHRRpWoWKtopoJoDA113drb0M5RkbQGoHvgEt3B4ORrOCqj8EpdsZFCYbrYsesi/MivLplhHzlhkhyGyw3+bmr4/Q2cEOpbU2NS9mb7p'
    '8YdT07Mz1NvEyzWH6ybli4uAh9QZ3dUqWOZmlZIMS16p8HGpBHZ6zwrR81wbel5ujI8wehx0wqB6ODU5ce+r3rHhL379Se6a+vHJ83e0LCGaaXO+iWbJqeuW'
    '3/t+083dXBNcqB9nOfv08Cr/wR2H7hcTYBm6W20TxnL1zhtdTrQANfF8CDYARaRsqoskCzg28QDx7kFesNaPGlFJiR6kyw5AGXlZFKj3vVSqkeklVheBZmdp'
    'vJwgEeSLrksBCUNT53oJ3oXLuS+5bQhEifew10FFBfCdoI2DqbvCrG3TCEjBbg9Qmb8cnZwAvMeHJ8fH/j4+3WmlLsB1gYF7iVvL2eW9OIyE+RphQtsQikbY'
    '3W4oHtqGi5WYgJC9y6uOWnZslKTt+hBbnloFPbPXJslIlKiVj5+QTQNLbri0IMkUwcbzGQ7zW9HRXBIEgTyu1oiQ4hp0pkZXLtnfB8SdwvWsPmMGZrQB8mXI'
    'UE+PpU9BGmAE70BsyrnH/mIPKKb6CUVuhUNEA0oNUbN+gYhZL4KwEfVKqFfWM5bDOH78GFrmwwB8SqvJ7pmEgL/IL2w2onbburBkdyI+0hhUvFZlWQ22dUts'
    'tF/N1DIT9GUO2pIix2KrBLeM2q0nV8WpWqhyZVuBNbBJwfHm83W7FN5FFtlEMVzozKKqmVC64DB41obVj0k1qcuwB/jBup1ruf6kaYNK9J+KTTQKwslF7KLI'
    '9QvABB2lMphNCOiyZV1s6xC7kk9ia6SuRLshI0iUvNsLEf5g+N0kC4zCc0BLJRjpexnwyVLqSJUiW4Yrgj3BNEl9I+CKrDUl6YCfRXanGl0ilzzYhZTEEIh5'
    'Q4q6vFdyVTdDgrpHXbS+TJST3MHrQL0yYSGZVcSSrRTEVXzKezjreoc4UtAjCD6bwwSffEYeyThGe3pCUA31Ku4DElQhssMwXwLji4F7NfVsXy/YLpGmte4m'
    'bViZT5SgmcrHoLN8k7bvBTRD7o2GFc61h1WztORX4chxzib4r5cVs93ARfzabLMpZ22lEwGCUZhP51UMpY+7JD3sDoKBewFphRgvCeTkdUnLwNvGL/f0iNKJ'
    '6mWGFCuQtqc6BHOeYB020fhdbyBZzwvoP6JRO3DgSXPkJTaX8MxZ5SHh8RwN8MA1+pUEDQ4Uz5G6Q19TY5xOarW0DVEqqCJBEFpMqhQyhmUoEybv9HNijYGk'
    'g2050lb0Bj4m7Ohd0OZHRAdxtDtDt/906/rdmzfSLwl5y/ssvtn+ZzAi+Ned1bK3z03T1j6Yl9+BpW3UFj00Zb+uU0N1afzZ8C3NEMClwbhDEHz1xM5o1A/U'
    '7kSGW7FLCdI/5IXziIREfDvvWtuPiSwImOnPk00yL9Hq8yEcS8q+QBS635ag1qZbpujrrwPbMkAZMQarVsOLDtvpRA/OYZRTjvTmUx5Yd950t6ttHW7NmgNw'
    '2lXJTL6viRyx7tXeFts4LBfJ5tIjU49mHz7CI5Pjo9MPxqeHRx+NTczm/jEz9aDrNuwxvPDMo8nZmfQm22XbsIB2b8ZzmNWKaX0HDNaXQU693KlhAX9nybg3'
    'GS8kPUDuYVu/PiY1znia7AQm5CGJoNlOtJQIXwwku4socMSsEqsQhpFoIEJYwkoFLSr8wg0p5xZLlGEakT1mBZiWRvYrFcYLZiUuDTNd5mORor4uehdrl50Y'
    'mLosIQgxU0DWfySlHsjxxz+xVoP/V5/GaEf63rBaYWVlzuiUj/ic/Rk6DGaQWW+DJze/TC2RTEVE5msLJFwwJDIeiRskaU1FknwvPcm+VHBwegi/5TlmzUm0'
    'QZLMzkuKnYp1z2VbBnZk7zX9UzmSMyL0rOWQU77NVz5Mce+e4t7myX2TbjV2k+9vrW9lZJJXgBHLGoi8T5gDr1hP3TFaNAK+cm6Nz4C0CAM8UAmF0w58LKt/'
    'mDX4o1Cm8lds5Rkhgx++kF30hczWNm1dy8RwqfCyQvlHLoqFxhON5eOwIPSJHqQzsAVip3ofTNoHk/Y/atLY5mCcWNvo64xuOLCdbLGY/cIiQxqX2rJdh1+u'
    'Cr+IjXgvZelVP23qetW7HV3n3lk71ytmru/8t1DiTCFJzgbCYOcSBc8RGY98QrkkpkcAtuRCviMRmniKJxKxHY2JD8Mf3k0mP0IUTX50IdlCNmSPgT1Ij0lD'
    'q3BH8VrBmqtOEnsy///6yX9/9cmS8l+w9J9gOfuxTT6nIF75XgcggZ7lIqmOkM9iRYg0Y9b/C1BLAQIUAxQAAAAIAAAAIVwHKwt0jQkAACcXAAASAAAAAAAA'
    'AAAAAACkAQAAAABTVFIwMV9NMDdfZm9ybXMubWRQSwECFAMUAAAACAAAACFcgfXTDwcLAACnHQAAEwAAAAAAAAAAAAAApAG9CQAAU1RSMDFfTTA3X2xlc3Nv'
    'bi5tZFBLBQYAAAAAAgACAIEAAAD1FAAAAAA='
)
_EMBEDDED_ZIP_BYTES = _embedded_b64.b64decode(_EMBEDDED_ZIP_B64)
print('📦 노트북에 내장된 실습자료를 준비합니다. ZIP 업로드는 필요하지 않습니다.')
_prepare_project_zip(_EMBEDDED_ZIP_BYTES)


## 미션 핵심 개념과 원본 실습 설명

---

## 🎯 미션 07. 결핍 부대 긴급 물자 전환 결정을 위한 군수처장 HITL 승인 워크플로우 설계

**소요시간**: 약 3.5시간

### 📋 미션 배경 및 수행 지침

#### 미션 배경 및 당위성

- 결핍 부대 긴급 물자 전환 결정을 위한 군수처장 HITL 승인 워크플로우를 설계한다.
- 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **예측 기반 보급 정책의 효과를 시뮬레이션으로 비교하고, 결핍 부대에 대한 긴급 물자 전환은 군수처장 승인으로만 집행하는 HITL 워크플로우를 설계한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.
- 이 문제를 해결하기 위해 본 미션에서는 **고정 목표 재고 정책과 예측 기반 동적 목표 정책을 같은 50일 자료로 재현해 결핍량, 재고 회전, 비용을 비교 수행하고 검증하는** 실습을 체계적으로 수행합니다.
- 학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/optimization/str01_simulator.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)

#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 현실적 군수 제약조건 반영도 및 시나리오별 대안 제시 적절성

#### 미션 PASS 조건

- 정량 기준: 최적화 적용 시 비용 15% 절감 또는 재고 회전율 20% 향상

- 진행 기준: 선형/정수 계획법(LP/MIP) 최적화 해 도출 및 시뮬레이션 완료

- 검증 방법: 과거 실적 데이터 대비 최적화 결과 비교 백테스팅

#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.


## 교관용 검증 관점

### AI 활용 예시 · 미션 07 전용

> 나는 Python을 처음 배우는 학습자야. **결핍 부대 긴급 물자 전환 결정을 위한 군수처장 HITL 승인 워크플로우 설계**을 진행하고 있어. 다음 요구사항을 보고 핵심 함수의 입력·출력, 필요한 검증 3개를 쉬운 말로 설명해줘. `TODO` 구간만 구현할 수 있는 코드를 제안하고, 데이터를 미리 보지 않는 검증 방법과 경계 사례도 알려줘. 실행 결과는 만들지 말고 내가 실제 실행한 로그를 받은 다음 분석해줘.  
> 요구사항: src/optimization/str01_simulator.py / simulate: policy,d_b=6.5 → 정책 시뮬레이션 집계 사전. A 고정/B 동적 목표를 동일50일 자료·4일 주기·가상 비용으로 비교한다. 잘못된 정책·유효하지 않은 d_b는 거부하고0분모는 산출불가다. 실제 물자 이동은 실행하지 않는다.

**AI 답변을 받으면:** 그대로 복사하지 말고 입력 시점·자료 범위·실제 실행 여부를 직접 확인하세요. 확인되지 않은 수치는 `미측정`, 수행하지 않은 활동은 `미실시`로 남깁니다.

원본 Solution 코드를 실행해 출력값과 조건을 대조하세요. 이 파일은 학생용에 배포하지 않습니다.

In [ ]:
from pathlib import Path

if not (globals().get('STR01_PROJECT_READY', False) and globals().get('STR01_PROJECT_DIR') and Path(STR01_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    """예측 기반 동적 목표 재고 정책을 고정 목표 정책과 시뮬레이션으로 비교해 비용·재고 회전율 효과를 구하고, 결핍 예상 부대에 대한 긴급 물자 전환 제안을 만들며 최종 집행은 군수처장 승인으로만 이루어지는 HITL 승인 워크플로우와 감사 이력·우회 차단을 구현한다."""
    import hashlib
    import json
    import numpy as np
    import pandas as pd
    from sklearn.ensemble import GradientBoostingRegressor

    # ---------------------------------------------------------------------------
    # [교육용가상자료] 가상 부대 20개 x 핵심 보급품 5종 일일 재고·소모 시뮬레이터 (실제 부대·수량·기준일수 아님)
    # 핵심성과지표(KPI) = 기준 보급일수(DOS) = 현재 재고 / 최근 평균 일일 소모량. 목표 DOS는 교육용 가정값이다.
    # 결핍 위험 일수 H = 보급이 없다고 가정할 때 현재 재고가 바닥나기까지 남은 일수(미래 실제 소모량으로 계산한 정답).
    # ---------------------------------------------------------------------------
    ITEMS = {"식량": (100, 0.3, 10), "피복": (20, 0.1, 15), "유류": (300, 1.2, 7), "탄약": (60, 1.5, 6), "수리부속": (15, 0.5, 12)}   # (기본 일소모, 템포 민감도, 목표 DOS)
    N_UNIT, DAYS = 20, 160
    UNITS = [f"가상{u:02d}부대" for u in range(1, N_UNIT + 1)]


    def make_data(seed=1, n_unit=N_UNIT, days=DAYS):
        """일 단위 재고 원장: unit, item, day, tempo(실제 작전 템포), plan(사전 계획 템포), wx(악천후), consume, resupply, stock."""
        rng = np.random.default_rng(seed)
        rows = []
        for u in range(1, n_unit + 1):
            s, ph = rng.uniform(0.7, 1.5), rng.uniform(0, 6.28)
            surge = np.zeros(days)
            for st in rng.choice(days - 10, 4, replace=False):
                surge[st:st + int(rng.integers(5, 10))] = rng.uniform(0.4, 0.8)
            tempo = 1 + 0.25 * np.sin(2 * np.pi * np.arange(days) / 30 + ph) + surge
            plan = tempo + rng.normal(0, 0.08, days)
            wx = (rng.random(days) < 0.15).astype(int)
            for it, (b, sens, tg) in ITEMS.items():
                stock, off = tg * b * s * rng.uniform(0.8, 1.2), int(rng.integers(0, 4))
                for t in range(days):
                    c = b * s * (1 + sens * (tempo[t] - 1)) * (1 + 0.12 * wx[t]) * rng.normal(1, 0.08)
                    r = max(tg * b * s - stock, 0) * rng.uniform(0.9, 1.1) * (0.5 if rng.random() < 0.08 else 1) if (t % 4 == off and t > 0) else 0.0   # 4일 주기 정기 보급: 목표 재고까지 채움(가끔 수송 차질로 절반)
                    stock = max(stock + r - c, 0.0)
                    rows.append((f"가상{u:02d}부대", it, t, tempo[t], plan[t], wx[t], c, r, stock))
        return pd.DataFrame(rows, columns=["unit", "item", "day", "tempo", "plan", "wx", "consume", "resupply", "stock"])


    def make_samples(d, lo=21, hi=125):
        """(부대, 품목, 기준일 t0)별 특성(피처)과 정답 H. 피처는 t0 시점까지 알 수 있는 값과 사전 계획·기상 예보만 사용한다."""
        out = []
        for (u, it), g in d.groupby(["unit", "item"]):
            c, stk, rs = g["consume"].values, g["stock"].values, g["resupply"].values
            pl, wx, tp = g["plan"].values, g["wx"].values, g["tempo"].values
            cs = np.cumsum(c)
            tg = ITEMS[it][2] * ITEMS[it][0]
            for t0 in range(lo, hi):
                idx = np.searchsorted(cs, cs[t0] + stk[t0])
                if idx >= len(cs) or idx <= t0:
                    continue
                c3, c7, c14 = c[t0 - 2:t0 + 1].mean(), c[t0 - 6:t0 + 1].mean(), c[t0 - 13:t0 + 1].mean()
                last = [k for k in range(t0, t0 - 15, -1) if rs[k] > 0]
                out.append({"unit": u, "item": it, "t0": t0, "H": idx - t0, "stock": stk[t0], "c3": c3, "c7": c7, "c14": c14,
                            "dos7": stk[t0] / c7, "dos3": stk[t0] / c3, "trend": c7 / c14, "burst": c3 / c7,
                            "plan7": pl[t0 + 1:t0 + 8].mean(), "plan_ratio": pl[t0 + 1:t0 + 8].mean() / tp[t0], "tempo0": tp[t0],
                            "wx_fc": wx[t0 + 1:t0 + 8].mean(), "since_rs": (t0 - last[0]) if last else 15,
                            "rs_avg": rs[t0 - 13:t0 + 1].sum() / 14, "fill": stk[t0] / tg, "item_code": list(ITEMS).index(it)})
        return pd.DataFrame(out)


    def mape(a, b):
        return float(np.mean(np.abs(a - b) / a))

    d = make_data()
    s = make_samples(d)
    X = [c for c in s.columns if c not in ("unit", "item", "t0", "H")]
    tr = s[s["t0"] <= 90]
    gbr = GradientBoostingRegressor(n_estimators=120, max_depth=3, learning_rate=0.1, random_state=0).fit(tr[X], tr["H"])
    now = s[s["t0"] == 100].copy()
    now["pred"] = np.clip(gbr.predict(now[X]), 1, None)
    POS = {u: np.array([i % 5, i // 5]) * 20.0 for i, u in enumerate(UNITS)}       # 가상 좌표(실제 위치 아님)
    T0, T1 = 100, 150

    # 1) 시뮬레이터: 100~149일을 실제 소모·계획·기상 자료로 재현해 두 가지 보급 정책을 비교(4일 주기 정기 보급)
    def simulate(policy, d_b=6.5):
        """정책 A: 고정 목표 재고(현행). 정책 B: 예측 기반 동적 목표(다음 4일 계획 템포·기상 예보를 반영한 소모 예상 x d_b일치)."""
        tot = {"short": 0.0, "stock": 0.0, "cons": 0.0, "ship": 0.0, "n": 0}
        for (u, it), g in d.groupby(["unit", "item"]):
            c, pl, wx, rs = g["consume"].values, g["plan"].values, g["wx"].values, g["resupply"].values
            b, sens, tg = ITEMS[it]
            stock = g["stock"].values[T0 - 1]
            off = [t for t in range(1, 30) if rs[t] > 0][0] % 4
            base_tg = tg * float(np.mean(c[:21]))
            for t in range(T0, T1):
                if t % 4 == off:
                    if policy == "A":
                        target = base_tg
                    else:
                        pr = pl[t:t + 4].mean() / max(pl[t - 7:t].mean(), 0.3)
                        f = c[t - 7:t].mean() * (1 + sens * (pr - 1)) * (1 + 0.12 * wx[t:t + 4].mean()) / (1 + 0.12 * wx[t - 7:t].mean())
                        target = d_b * f
                    r = max(target - stock, 0); stock += r; tot["ship"] += r
                use = min(stock, c[t]); tot["short"] += c[t] - use; stock -= use; tot["stock"] += stock; tot["cons"] += use; tot["n"] += 1
        avg = tot["stock"] / tot["n"]
        tot["turn"] = tot["cons"] / avg / (T1 - T0)                       # 재고 회전율(연간 환산 아님: 일평균 소모/평균 재고의 합계 기준 상대 지표)
        tot["short_rate"] = tot["short"] / (tot["cons"] + tot["short"])
        tot["cost"] = 0.02 * tot["stock"] + 8.0 * tot["short"]            # 가정: 보유비 0.02/재고일, 결핍 손실 8.0/단위
        return tot
    A, B = simulate("A"), simulate("B")
    print(f"[보급 정책 시뮬레이션 {T0}~{T1 - 1}일] 부대 20 x 품목 5 [교육용가상자료]")
    print(f"{'':<16}{'A 고정 목표 재고':>18}{'B 예측 기반 동적 목표':>22}")
    for k, nm, fmt in (("short", "결핍량 합계", "{:,.0f}"), ("short_rate", "결핍률", "{:.2%}"), ("stock", "재고일 합계", "{:,.0f}"), ("turn", "재고 회전 지표", "{:.2f}"), ("cost", "비용(가정 단가)", "{:,.0f}")):
        print(f"{nm:<14}{fmt.format(A[k]):>20}{fmt.format(B[k]):>22}")
    sav, turn_gain = 1 - B["cost"] / A["cost"], B["turn"] / A["turn"] - 1
    print(f"재고 회전율 향상 {turn_gain:.1%}, 비용 절감 {sav:.1%}(보유비+결핍 손실, 가정 단가), 결핍량 감소 {1 - B['short'] / A['short']:.1%}")
    print(f"목표(비용 15% 절감 또는 회전율 20% 향상): {'달성' if sav >= 0.15 or turn_gain >= 0.2 else '미달'}")
    print("제약: 4일 주기 정기 보급, 보급량은 목표 재고까지, 결핍은 이월하지 않음(가정). 시나리오 A와 B 중 B가 결핍을 줄이면서 재고를 줄이는지 함께 확인한다.")

    # 2) 긴급 전환 제안 생성: 예측 결핍(경계 이하) 부대에 다음 보급일까지 버틸 물량을 여유 부대에서 옮기거나 보급소 예비로 충당
    now["nxt"] = (3 - now["since_rs"]).clip(lower=0) + 1
    now["keep"] = 1.3 * now["c7"] * (now["nxt"] + 1)
    donors = now[(now["pred"] > 6) & (now["stock"] > now["keep"])].copy(); donors["avail"] = donors["stock"] - donors["keep"]
    plan = []
    for r in now[now["pred"] <= 4.5].sort_values("pred").itertuples():
        q = r.keep - r.stock
        if q <= 0:
            continue
        cand = donors[(donors["item"] == r.item) & (donors["avail"] >= q)]
        if len(cand):
            j = cand.assign(dist=[np.linalg.norm(POS[u] - POS[r.unit]) for u in cand["unit"]]).sort_values("dist").iloc[0]
            donors.loc[(donors["unit"] == j["unit"]) & (donors["item"] == j["item"]), "avail"] -= q
            plan.append((j["unit"], r.unit, r.item, float(q)))
        else:
            plan.append(("보급소 예비", r.unit, r.item, float(q)))
    n_lat = sum(m[0] != "보급소 예비" for m in plan)
    print(f"\n[긴급 전환 제안 - {T0}일] 결핍 예상(경계 이하) {int((now['pred'] <= 4.5).sum())}건 중 제안 {len(plan)}건(부대 간 전환 {n_lat}건, 보급소 예비 {len(plan) - n_lat}건)")
    for m in plan[:4]:
        print(f"  {m[0]} -> {m[1]} {m[2]} {m[3]:.0f}")

    # 2) HITL: 군수처장 승인 워크플로우(자동 집행 금지)
    USERS = {"bot": "시스템", "kim": "군수담당", "lee": "군수담당", "jang": "군수처장", "han": "열람자"}
    AUDIT, PROP = [], {}
    def h(prev, rec):
        return hashlib.sha256((prev + json.dumps(rec, ensure_ascii=False, sort_keys=True)).encode()).hexdigest()[:16]
    def log(pid, user, act, res, note=""):
        rec = {"seq": len(AUDIT) + 1, "pid": pid, "user": user, "role": USERS.get(user, "미등록"), "act": act, "res": res, "note": note}
        rec["hash"] = h(AUDIT[-1]["hash"] if AUDIT else "G", {k: v for k, v in rec.items() if k != "hash"}); AUDIT.append(rec)
    def blocked(pid, user, act, why):
        log(pid, user, act, "차단", why); return False
    def propose(pid, user, move):
        if USERS.get(user) not in ("시스템", "군수담당"):
            return blocked(pid, user, "제안", "제안 권한 없음")
        PROP[pid] = {"move": move, "by": user, "status": "대기"}; log(pid, user, "제안", "접수", f"{move[0]}->{move[1]} {move[2]} {move[3]:.0f}"); return True
    def decide(pid, user, act, note):
        p = PROP.get(pid)
        if p is None:
            return blocked(pid, user, act, "제안 없음")
        if USERS.get(user) != "군수처장":
            return blocked(pid, user, act, f"권한 없음: 역할 '{USERS.get(user, '미등록')}'은(는) 결정 불가(자동 집행 금지)")
        if p["status"] != "대기":
            return blocked(pid, user, act, f"이미 {p['status']}")
        if len(note.strip()) < 5:
            return blocked(pid, user, act, "결정 근거 5자 이상 필요")
        p["status"] = "승인" if act == "승인" else "반려"; log(pid, user, act, "완료", note); return True
    def execute(pid, user):
        p = PROP.get(pid)
        if p is None or p["status"] != "승인":
            return blocked(pid, user, "집행", "승인되지 않은 건 집행 불가")
        p["status"] = "집행"; log(pid, user, "집행", "완료", "승인 건 집행(모의)"); return True

    big = plan[:8]
    for i, mv in enumerate(big):
        propose(f"P{i + 1}", "bot" if i % 2 else "kim", mv)
    if len(plan) > 8:
        propose("P9", "kim", plan[8])                          # 결정 전 대기 건(우회 시험용)
    for i, mv in enumerate(big):
        pid = f"P{i + 1}"
        if i == 5:
            decide(pid, "jang", "반려", "여유 부대의 다음 훈련 일정과 충돌 우려")
        else:
            decide(pid, "jang", "승인", "예측 근거와 여유 부대 5일치 유지 확인")
        execute(pid, "kim")
    att = [("시스템이 자동 승인", lambda: decide("P1", "bot", "승인", "자동 승인 처리")), ("군수담당이 직접 승인", lambda: decide("P2", "lee", "승인", "담당이 직접 승인함")),
           ("열람자가 승인", lambda: decide("P3", "han", "승인", "열람자가 승인 시도함")), ("미등록 사용자 승인", lambda: decide("P4", "x", "승인", "외부 계정 승인 시도")),
           ("승인 근거 없이 결정", lambda: decide("P9", "jang", "승인", "")), ("반려 건 집행", lambda: execute("P6", "kim")),
           ("미승인 건 집행(없는 제안)", lambda: execute("P99", "kim")), ("이미 집행된 건 재승인", lambda: decide("P1", "jang", "승인", "중복 승인 시도 확인"))]
    print("\n[HITL 승인 결과]", {k: sum(p["status"] == k for p in PROP.values()) for k in ("집행", "반려", "대기")})
    nb = 0
    for name, fn in att:
        n = len(AUDIT); fn(); r = AUDIT[-1]; ok = len(AUDIT) == n + 1 and r["res"] == "차단"; nb += ok
        print(f" - {name}: {r['res']} ({r['note']})")
    prev, okc = "G", 0
    for r in AUDIT:
        okc += r["hash"] == h(prev, {k: v for k, v in r.items() if k != "hash"}); prev = r["hash"]
    print(f"우회 시도 {len(att)}건 중 차단 {nb}건, 감사 이력 {len(AUDIT)}건 해시 검증 {okc}/{len(AUDIT)}")
    print("군수처장 승인 없이 집행된 건 0건. 실제 물자 전환 결정은 지휘계통과 군수 규정에 따르며 본 코드는 학습용 모의이다.")


## 원본 교관용 예상 출력 — 실측이 아닌 참고 예시

<details>
<summary><b>💡 [참고] 미션 07 예상 실행 결과 (Output)</b></summary>

```text
[보급 정책 시뮬레이션 100～149일] 부대 20 x 품목 5 [교육용가상자료]
                        A 고정 목표 재고         B 예측 기반 동적 목표
결핍량 합계                       5,546                   894
결핍률                          0.89%                 0.14%
재고일 합계                   3,317,341             2,505,060
재고 회전 지표                     18.54                 24.74
비용(가정 단가)                  110,716                57,250
재고 회전율 향상 33.4%, 비용 절감 48.3%(보유비+결핍 손실, 가정 단가), 결핍량 감소 83.9%
목표(비용 15% 절감 또는 회전율 20% 향상): 달성
제약: 4일 주기 정기 보급, 보급량은 목표 재고까지, 결핍은 이월하지 않음(가정). 시나리오 A와 B 중 B가 결핍을 줄이면서 재고를 줄이는지 함께 확인한다.

[긴급 전환 제안 - 100일] 결핍 예상(경계 이하) 19건 중 제안 17건(부대 간 전환 3건, 보급소 예비 14건)
  보급소 예비 -> 가상01부대 탄약 195
  보급소 예비 -> 가상19부대 탄약 211
  보급소 예비 -> 가상15부대 탄약 238
  보급소 예비 -> 가상03부대 탄약 582

[HITL 승인 결과] {'집행': 7, '반려': 1, '대기': 1}
 - 시스템이 자동 승인: 차단 (권한 없음: 역할 '시스템'은(는) 결정 불가(자동 집행 금지))
 - 군수담당이 직접 승인: 차단 (권한 없음: 역할 '군수담당'은(는) 결정 불가(자동 집행 금지))
 - 열람자가 승인: 차단 (권한 없음: 역할 '열람자'은(는) 결정 불가(자동 집행 금지))
 - 미등록 사용자 승인: 차단 (권한 없음: 역할 '미등록'은(는) 결정 불가(자동 집행 금지))
 - 승인 근거 없이 결정: 차단 (결정 근거 5자 이상 필요)
 - 반려 건 집행: 차단 (승인되지 않은 건 집행 불가)
 - 미승인 건 집행(없는 제안): 차단 (승인되지 않은 건 집행 불가)
 - 이미 집행된 건 재승인: 차단 (이미 집행)
우회 시도 8건 중 차단 8건, 감사 이력 33건 해시 검증 33/33
군수처장 승인 없이 집행된 건 0건. 실제 물자 전환 결정은 지휘계통과 군수 규정에 따르며 본 코드는 학습용 모의이다.
```
</details>


## 📋 3. 학습활동 4단계

### 활동 1: 무엇을 확인하고 계획하나요?

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 원본 목표, 기본 코드, 그리고 제공된 학습 자료를 면밀히 분석하여, "비용 절감에도 결핍이 늘면 어떤 정책을 선택할 거야?"라는 핵심 질문에 대해 사전에 엄밀히 판단합니다.
  - 아울러 예측 기반 보급 정책의 효과를 시뮬레이션으로 비교하고, 결핍 부대에 대한 긴급 물자 전환을 군수처장 승인으로만 집행하는 HITL(Human-in-the-Loop) 워크플로우를 설계하는 과정에서 AI 에이전트에 위임할 분석 범위, 데이터 제약 사항, 입력 및 기간·단위·사용 가능 시점, 그리고 simulate 함수의 입출력 계약 조건을 체계적으로 수립합니다.
- **수행 방식:**
  - 막연한 코딩이나 AI의 무분별한 답변에 의존하지 않고, 군수 의사결정의 경제성과 안전 재고 유지 간의 균형을 잡기 위한 명확한 사전 규칙과 검증 계획을 문서화합니다.

#### 2. 세부 수행 절차

- **원본 목표·기본 코드·자료 분석 및 핵심 질문 검토:**
  - 예측 기반 보급 정책의 효과를 시뮬레이션하고 긴급 물자 전환을 군수처장 HITL 승인으로 집행하는 데 필요한 입력 데이터, 기간, 단위, 그리고 사용 가능한 시점의 제약 조건을 파악합니다.
  - 단순한 회계적 비용 절감 효과가 있더라도 전술적 결핍 리스크가 증가할 경우 어떤 보급 정책(고정 목표 vs 동적 목표)을 선택해야 하는지 사전에 분석하고 정의합니다.
- **simulate 입출력 계약 정의:**
  - 정책 유형 및 동적 목표 일수 등의 파라미터를 입력받아 정책 시뮬레이션 결과를 집계하는 핵심 함수 simulate 의 입출력 계약 조건을 명확히 이해합니다.
- **AI 위임 범위 및 검증 계획 정의:**
  - AI 에이전트에 맡길 코드 구현 및 분석 범위, 데이터 무결성 검증 방법을 설정하고, 본인의 사전 판단과 검증 계획을 서식( forms/STR01_M07_forms.md )에 명시합니다.

#### 3. 주의 및 검…

#### 완료 확인:

- 이 활동의 확인 근거 M07 계획: 예측 기반 보급 정책의 효과를 시뮬레이션으로 비교하고, 결핍 부대에 대한 긴급 물자 전환은 군수처장 승인으로만 집행하는 HITL 워크플로우를 설계한다.에 필요한 입력·기간·단위·사용 가능 시점과 simulate의입출력 계약을 설명하고 본인의사전판단·AI위임·검증계획을 기록했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 2: AI와 코드 초안을 만들어 실행하기

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 핵심 함수인 simulate 를 구현·저장하고 첫 성공 실행 결과를 온전히 보존합니다.
  - 정책 및 동적 목표 일수 계약조건( policy, d_b = 6.5 → 정책 시뮬레이션 집계 사전, A 고정/B 동적 목표를 동일 50일 자료·4일 주기·가상 비용으로 비교)을 준수하며, 잘못된 정책이나 유효하지 않은 d_b 값은 거부하고 0분모인 경우 산출이 불가하도록 처리합니다.
  - 단, 실제 물자 이동이나 자동 집행은 실행하지 않습니다.
  - 아울러 변경 과제(동적 목표 일수 d_b 를 6.5에서 5.0으로 변경하여 비용·결핍·회전을 비교)를 수행하고, 별도의 직접 입력 도구로 제안 3건 이상을 승인·반려·보류하며 그 근거를 기록합니다.
- **수행 방식:**
  - AI와 협업하여 정책 시뮬레이션 및 HITL 가상 승인 로직을 코드에 반영하고, 기준 코드 실행 및 변경 실험을 수행하여 실제 코드와 출력 결과를 정확하게 확보합니다.

#### 2. 세부 수행 절차

- 핵심 함수 simulate 구현 및 첫 성공 실행: policy 와 d_b 등의 인자를 받아 고정 목표(A)와 동적 목표(B) 정책을 동일한 50일 자료, 4일 주기, 가상 비용 기준으로 비교 집계하는 simulate 함수를 구현합니다.
- 유효하지 않은 정책 유형이나 잘못된 d_b 값은 예외 처리로 거부하고, 분모가 0인 산출 불가 상황을 엄격히 통제합니다.
- 실제 물리적인 물자 이동이나 자동 집행은 수행하지 않음을 명시하고, 첫 성공 실행 결과, 로그, 표 및 그림 형태의 출력을 누락 없이 보존합니다.
- 변경 과제 수행 (동적 목표 일수 변경: d_b = 6.5 → 5.0 ): 동적 목표 일수 파라미터 d_b 를 기존 6.5에서 5.0으로 변경하여 코드를 수정하고 재실행합니다.
- 변경에 따른 비용, 결핍 리스크, 그리고 재고 회전 지표의 변화 양상을 비…

#### 완료 확인:

- 이 활동의 확인 근거 M07 구현: src/optimization/str01_simulator.py의 simulate를저장하고계약〔policy,d_b=6.5 → 정책 시뮬레이션 집계 사전.
- A 고정/B 동적 목표를 동일50일 자료·4일 주기·가상 비용으로 비교한다.
- 잘못된 정책·유효하지 않은 d_b는 거부하고0분모는 산출불가다.
- 실제 물자 이동은 실행하지 않는다.〕을충족하는 첫성공실행·로그·표/그림을보존했는가?
- 이후Lesson의추가변경을실제코드에반영했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 3: 원자료와 결과를 대조·수정하기

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 앞서 구현 및 수정된 정책 시뮬레이션 및 HITL 워크플로우 코드를 저장하고 재실행합니다.
  - 정책 적용 시 비용 15% 이상 절감 또는 재고 회전율 20% 이상 향상 효과를 입증하고, AI의 자동 결정, 권한 없는 결정, 무사유·중복 결정, 자원 초과 할당 등을 철저히 차단했는지 검증합니다.
  - 아울러 AI의 설명 및 코드 선택 3건 이상을 실제 군수 도메인 자료 및 실행 결과와 철저히 대조하여 보완된 답변을 도출합니다.
- **수행 방식:**
  - 단편적인 실행에 그치지 않고, 동일한 50일 자료, 4일 주기, 가상 비용 조건 및 실제 기간·분모·조건·수치를 바탕으로 최초 실행 결과와 수정 실행 결과를 비교 대조하며 AI의 답변을 비판적으로 교차 검증합니다.

#### 2. 세부 수행 절차

- **수정 코드 저장 및 재실행:**
  - 정책 시뮬레이션 소스 코드( src/optimization/str01_simulator.py )를 안전하게 저장하고 재실행하여 에러 여부와 실행 로그를 확보합니다.
  - 정량적 목표 및 군수 제약 준수 판정: 고정 목표(A)와 동적 목표(B) 정책을 동일한 50일 자료로 비교하여, 최적화 적용 시 비용 15% 이상 절감 또는 재고 회전율 20% 이상 향상 효과가 나타나는지 정량적으로 판정합니다.
  - 보급 주기 및 안전 재고 유지 등 군수 제약 조건이 시나리오별 대안에 적절히 반영되었는지, 그리고 시스템이 임의로 물자를 자동 집행하지 않고 수동 검토 체계를 유지하는지 확인합니다.
- **부적절한 자동화 및 자원 충돌 차단 검증:**
  - AI가 임의로 결정을 내리거나, 권한 없는 주체가 승인하는 행위, 사유가 없는 결정, 중복 결정, 자원 한도를 초과하는 예약 시도가 올바르게 차단되는지 점검합니다.
  - 제안 시 자원을 예약하는 보수적 모형의 특성상 반려 후 자원이 자동으로 재배정되지 않는지 규칙을 엄밀히 대…

#### 완료 확인:

+ **수행 조건과 판정 기준**
  - 이 활동의 확인 근거 M07 검증: [필수] 정책 시뮬레이션·군수처장 HITL — 최적화 적용 시 비용 15% 절감 또는 재고 회전율 20% 향상을 보이고, 군수 제약(보급 주기, 안전 재고 유지)과 시나리오별 대안이 적절하게 반영되며 자동 집행이 없는가.
  - 실제기간·분모·조건·수치로판정하고 최초/수정실행을비교했는가?

+ **미션별 검증 조건 · 측정 조건과 결과 해석**
  - 50일 동일 자료로 고정/동적 정책을 비교합니다.
  - 서로 다른 품목 수량을 합친 회전은 원본의 상대 지표이며 회계상 회전율이 아닙니다.
  - 단가는 가정이고 예측 오류·결핍·재고 상충을 함께 봅니다.
  - 참고 코드의 SIMULATED 결정은 학습자 직접 입력과 다릅니다.

+ **미션별 검증 조건 · 판정 상태와 적용 범위**
  - 직접 입력은 군수처장 가상 역할이며 실제 인증·물자 이동이 아닙니다.
  - 제안 시 자원을 예약하는 보수적 모형이므로 반려 후 자원을 자동 재배정하지 않습니다.
  - 해시는 내부 변경 검사이고 외부 신…

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 4: 최종 결과와 수행 상태 기록하기

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 미션 7(결핍 부대 긴급 물자 전환 결정을 위한 군수처장 HITL 승인 워크플로우 설계) 전 과정을 수행하며 산출된 정책 시뮬레이션 소스 코드 , 정책 분석 보고서 , 학습자 감사 파일 , 예측 결과 데이터 파일 , 그리고 실습 기록 서식 을 누락 없이 집대성합니다.
  - 아울러 정량적 목표 달성 여부, 미측정 영역, 문서 작성 상태 및 교관(또는 시스템) 승인 대기 상태를 명확히 구분하여 최종 판단을 기록합니다.
- **수행 방식:**
  - 단편적인 파일 제출에 치중하지 않고, 이번 미션에서 구축한 고정/동적 목표 정책 비교 결과와 군수처장 가상 HITL 수동 승인 워크플로우의 정당성을 객관적으로 검수하여 후속 미션(미션 8: 전면전 발발 모의 전 부대 보급품 동시 급감 스트레스 시뮬레이션)으로 온전히 인계할 수 있도록 패키지를 정비합니다.

#### 2. 세부 수행 절차

- **핵심 산출 파일 패키지 확정 및 제출:**
  - 정책 시뮬레이션 소스 코드인 src/optimization/str01_simulator.py , 정책 분석 보고서인 reports/str01_policy_report.md , 학습자 감사 파일인 outputs/learner_audit.json , 처리 결과 데이터인 outputs/M07/results.json , 그리고 실습 기록 서식인 forms/STR01_M07_forms.md 의 실제 작성 파일들을 최종 점검하고 제출 상태로 정리합니다.
- **코드 실행 결과 및 AI 대화 기록 연계 보존:**
  - simulate 함수 구현 및 고정/동적 정책 비교 결과, 그리고 분석 과정에서 AI 에이전트와 나눈 핵심 질의응답 및 코드 선택 근거 대화 내용을 실습 대화 보존 위치에 명확히 연결합니다.
- **상태별 최종 판단 기록 작성:**
  - 정량적 목표 달성 여부, 현장 여건상 미처 측정하지 못한 영역, 문서 작성 완…

#### 완료 확인:

- 이 활동의 확인 근거 M07 완료: src/optimization/str01_simulator.py, reports/str01_policy_report.md, outputs/learner_audit.json, outputs/M07/results.json, forms/STR01_M07_forms.md의실제작성·최종값·대화보존위치·선택이유가연결되는가?
- 미달·미측정·문서미작성·실제승인대기를정확히표시했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:


## 🔍 교관 평가 기준

- [정량적 평가] [필수] 정책 시뮬레이션·군수처장 HITL — 최적화 적용 시 비용 15% 절감 또는 재고 회전율 20% 향상을 보이고, 군수 제약(보급 주기, 안전 재고 유지)과 시나리오별 대안이 적절하게 반영되며 자동 집행이 없는가. PASS 증거 제공된조건·단위·기간·분모와실제수치/건수/항목근거를연결합니다. 목표미달·0기준·미측정은그대로표시합니다. 보완 방법 기준별빠진항목과미달원인을찾아수정하고재검증합니다.
- **[정성적 평가] AI 활용: 본인 계획·실제 대화·코드 변경·검토3건·보완·재실행·전후 비교·최종 선택.**
  - **PASS 증거**
    - **수행 조건과 판정 기준 · 측정 조건과 결과 해석**
      - 50일 동일 자료로 고정/동적 정책을 비교합니다.
      - 서로 다른 품목 수량을 합친 회전은 원본의 상대 지표이며 회계상 회전율이 아닙니다.
      - 단가는 가정이고 예측 오류·결핍·재고 상충을 함께 봅니다.
      - 참고 코드의 SIMULATED 결정은 학습자 직접 입력과 다릅니다.
    - **수행 조건과 판정 기준 · 판정 상태와 적용 범위**
      - 직접 입력은 군수처장 가상 역할이며 실제 인증·물자 이동이 아닙니다.
      - 제안 시 자원을 예약하는 보수적 모형이므로 반려 후 자원을 자동 재배정하지 않습니다.
      - 해시는 내부 변경 검사이고 외부 신뢰 기준점 없이 로그 전체 재작성은 막지 못합니다.
    - **보완 방법**
      - 보완 방법 근거없는원인·규정·완료추정은수정하거나확인필요로보류합니다.
- **[진행 추적 및 완료 기준] 정성·검증:**
  - **수행 조건과 판정 기준 · 측정 조건과 결과 해석**
    - 50일 동일 자료로 고정/동적 정책을 비교합니다.
    - 서로 다른 품목 수량을 합친 회전은 원본의 상대 지표이며 회계상 회전율이 아닙니다.
    - 단가는 가정이고 예측 오류·결핍·재고 상충을 함께 봅니다.
    - 참고 코드의 SIMULATED 결정은 학습자 직접 입력과 다릅니다.
  - **수행 조건과 판정 기준 · 판정 상태와 적용 범위**
    - 직접 입력은 군수처장 가상 역할이며 실제 인증·물자 이동이 아닙니다.
    - 제안 시 자원을 예약하는 보수적 모형이므로 반려 후 자원을 자동 재배정하지 않습니다.
    - 해시는 내부 변경 검사이고 외부 신뢰 기준점 없이 로그 전체 재작성은 막지 못합니다.
  - **제출 파일 및 완료 기록**
    - 파일: src/optimization/str01_simulator.py, reports/str01_policy_report.md, outputs/learner_audit.json, outputs/M07/results.json, forms/STR01_M07_forms.md 실제AI요청·답변·근거대조3건이상·보완·재확인·본인최종선택과작성/실행기록이연결됩니다.
    - 최종ZIP생성은실제학습완료·교관승인 인증이아닙니다.
  - **보완 방법**
    - 보완 방법 미작성·미실행·검토누락을보완하고교관검토상태를별도로표시합니다.
실제 수행·실행·사람의 승인 여부를 별도로 판정하세요.

**교관용 결과 저장:** 마지막 통합 화면에서 산출물을 개별 작성·수정·저장하고, 학습기록·실제 AI 대화·근거·3건 검토·최종 상태를 입력한 후 ZIP으로 저장할 수 있습니다.

교관용 참고 예시와 실제 실행 증거는 구분하세요.


In [ ]:
# @title ⑤ 교관용 개별 산출물·학습기록 통합 입력, 저장 및 ZIP 다운로드 (실행 버튼을 누르세요) { display-mode: "form" }
# 입력 폼이 나타나면 소스코드 대신 버튼을 이용하세요. Colab/Jupyter에서 ipywidgets 필요.
from pathlib import Path

if not (globals().get('STR01_PROJECT_READY', False) and globals().get('STR01_PROJECT_DIR') and Path(STR01_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    def _load_notebook_widgets():
        from importlib import import_module, invalidate_caches
        import subprocess
        import sys

        try:
            return import_module('ipywidgets')
        except ImportError:
            print('이 노트북 커널에 ipywidgets가 없어 설치합니다.')

        try:
            subprocess.run(
                [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                 '--retries', '0', '--timeout', '10', 'ipywidgets'],
                check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=60,
            )
            invalidate_caches()
            return import_module('ipywidgets')
        except (subprocess.SubprocessError, OSError, ImportError):
            print('이 커널에서 ipywidgets를 설치하거나 불러오지 못했습니다.')
            print('설치 가능한 환경에서는 새 셀에서 %pip install ipywidgets 를 실행한 뒤 이 셀을 다시 실행하세요.')
            print('설치할 수 없는 환경에서는 제공된 Markdown 작성 양식을 사용하세요.')
            return None

    widgets = _load_notebook_widgets()
    if widgets is not None:
        import json, csv, io, ast, zipfile, datetime, pathlib
        from pathlib import Path
        import os
        from urllib.parse import quote
        from IPython.display import HTML, display, Markdown, clear_output

        MISSION_CODE = 7
        DELIVERABLE_PATHS = ['src/optimization/str01_simulator.py', 'reports/str01_policy_report.md', 'outputs/learner_audit.json', 'outputs/M07/results.json', 'forms/STR01_M07_forms.md']
        PROJECT_NAME = 'STR01'
        NOTEBOOK_ROLE = '교관용 Solution'
        WORK_ROOT = Path(STR01_PROJECT_DIR).resolve()

        RECORD_DIR = WORK_ROOT / f'STR01_M{MISSION_CODE:02d}_학습기록'
        RECORD_DIR.mkdir(parents=True, exist_ok=True)
        ZIP_FILENAME = f'STR01_m{MISSION_CODE:02d}_review.zip'

        # 반드시 CMS 지정 경로만 사용하며 상대 경로의 작업공간 이탈을 방지합니다.
        def safe_path(relative):
            target = (WORK_ROOT / relative).resolve()
            if not target.is_relative_to(WORK_ROOT):
                raise ValueError('작업공간 밖의 경로는 사용할 수 없습니다.')
            return target

        def valid_text(path, content):
            suffix = Path(path).suffix.lower()
            if suffix == '.json':
                json.loads(content)
            elif suffix == '.ipynb':
                doc = json.loads(content)
                if not isinstance(doc, dict) or not isinstance(doc.get('cells'), list) or 'nbformat' not in doc:
                    raise ValueError('노트북 JSON에 cells 또는 nbformat 정보가 없습니다.')
            elif suffix == '.py':
                ast.parse(content)
            elif suffix == '.csv':
                if not content.strip():
                    raise ValueError('CSV 데이터가 비어 있습니다.')
                rows = list(csv.reader(io.StringIO(content)))
                if not rows or not rows[0] or not any(x.strip() for x in rows[0]):
                    raise ValueError('CSV 첫 행에 열 이름을 입력하세요.')
                if any(len(row) != len(rows[0]) for row in rows if row):
                    raise ValueError('CSV 열 개수가 행마다 다릅니다.')

        # 산출물 입력 화면 — 파일별 별도 탭
        editors = {}
        file_tabs = []
        file_buttons = {}
        for relative in DELIVERABLE_PATHS:
            suffix = Path(relative).suffix.lower()
            hint = {
                '.md': '보고서·양식의 제목, 분석 결과, 근거와 수정 이유를 Markdown으로 작성합니다.',
                '.csv': '첫 줄에 열 이름, 다음 줄부터 데이터 행을 쉼표로 구분하여 입력합니다.',
                '.json': '키와 값이 있는 유효한 JSON 형식으로 실제 결과를 입력합니다.',
                '.py': '직접 구현하고 확인한 Python 소스코드를 입력합니다.',
                '.ipynb': '노트북 JSON 원문을 입력하거나 기존 파일에서 불러옵니다. JSON 노트북 형식을 유지하세요.'
            }.get(suffix, '실제 작성한 내용을 입력합니다.')
            editor = widgets.Textarea(value='', placeholder='여기에 내용을 작성하세요. 입력하지 않은 파일은 저장하지 않습니다.',
                                      layout=widgets.Layout(width='100%', height='250px'))
            status = widgets.HTML('<span style="color:#777">아직 저장하지 않았습니다.</span>')
            load_btn = widgets.Button(description='기존 파일 불러오기', icon='folder-open', button_style='')
            save_btn = widgets.Button(description='이 파일 저장', icon='save', button_style='success')
            editors[relative] = {'editor': editor, 'status': status, 'loaded': False, 'saved': False, 'dirty': False, 'saved_value': ''}
            def on_edit(change, key=relative):
                state = editors[key]
                state['dirty'] = change['new'] != state['saved_value']
                state['saved'] = state['loaded'] and not state['dirty']
                if state['dirty']:
                    state['status'].value = '<span style="color:#a65">● 수정됨 — 저장 버튼을 누르세요.</span>'
                elif state['saved']:
                    state['status'].value = '<span style="color:#286">저장된 내용과 같습니다.</span>'
                else:
                    state['status'].value = '<span style="color:#777">아직 저장하지 않았습니다.</span>'
            editor.observe(on_edit, names='value')
            def load_clicked(button, key=relative):
                state = editors[key]
                target = safe_path(key)
                if not target.is_file():
                    state['status'].value = '<span style="color:#a22">파일이 아직 없습니다. 직접 입력 후 저장하세요.</span>'
                    return
                if state['dirty']:
                    state['status'].value = '<span style="color:#a22">미저장 내용이 있습니다. 불러오기 전에 저장하세요.</span>'
                    return
                try:
                    state['loaded'] = False
                    state['saved_value'] = target.read_text(encoding='utf-8')
                    state['editor'].value = state['saved_value']
                    state['loaded'] = True
                    state['dirty'] = False
                    state['saved'] = True
                    state['status'].value = '<span style="color:#286">파일을 불러왔습니다. 변경하면 다시 저장하세요.</span>'
                except Exception as exc:
                    state['status'].value = '<span style="color:#a22">불러오기 오류: '+str(exc)+'</span>'
            def save_clicked(button, key=relative):
                save_one(key)
            load_btn.on_click(load_clicked)
            save_btn.on_click(save_clicked)
            file_buttons[relative] = {'load':load_btn,'save':save_btn}
            file_tabs.append(widgets.VBox([
                widgets.HTML('<b>저장 경로:</b> <code>'+relative+'</code><br><span>'+hint+'</span><br><span style="color:#a55">기존 파일이 있다면 저장 시 덮어씁니다. 실습 완료·승인과는 다릅니다.</span>'),
                editor, widgets.HBox([load_btn, save_btn]), status
            ]))

        def save_one(relative):
            state = editors[relative]
            value = state['editor'].value
            if not value.strip():
                state['status'].value = '<span style="color:#a22">비어 있어 저장하지 않았습니다.</span>'
                return False
            try:
                valid_text(relative, value)
                target = safe_path(relative)
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_text(value, encoding='utf-8')
                state['saved_value'] = value
                (WORK_ROOT / ZIP_FILENAME).unlink(missing_ok=True)
                state['loaded'] = True
                state['dirty'] = False
                state['saved'] = True
                state['status'].value = '<span style="color:green">✓ 저장 완료: '+relative+'</span>'
                return True
            except Exception as exc:
                state['status'].value = '<span style="color:#a22">저장 실패: '+str(exc)+'</span>'
                return False

        # 학습기록 및 AI 원문, 자료 대조, 검토 3건 이상
        record_specs = [
         ('plan','학습 계획·수행 기록','이번 미션에서 직접 수행한 내용, 실행 조건 및 최초 결과를 작성하세요.'),
         ('ai_request','실제 AI 질문 원문','실제로 입력한 질문 원문만 작성하세요. 사용하지 않았다면 비워 둡니다.'),
         ('ai_response','실제 받은 AI 답변 원문','실제로 받은 답변 전체를 붙여 넣으세요. 임의로 생성한 대화는 기록하지 않습니다.'),
         ('evidence','실제 출력·자료 대조 근거','실제 출력값, 원자료 위치, 전후 비교 및 계산 근거를 적으세요.'),
         ('revision','검토 및 수정 내용 종합','검토 3건의 발견사항과 코드를 수정·재실행한 기록을 종합하세요.'),
         ('decision','최종 판단과 미달·미측정·미실시 사유','달성 여부와 남은 부분, 근거 및 추가 확인사항을 적으세요.'),
        ]
        record_widgets = {}
        record_items = []
        for key, title, hint in record_specs:
            field = widgets.Textarea(placeholder=hint, layout=widgets.Layout(width='100%',height='115px'))
            record_widgets[key] = field
            record_items.append(widgets.VBox([widgets.HTML('<b>'+title+'</b><br><small>'+hint+'</small>'),field]))

        reviews = []
        for i in range(3):
            claim = widgets.Textarea(placeholder=f'{i+1}번째 AI 주장·코드 선택 또는 점검 대상',layout=widgets.Layout(width='100%',height='65px'))
            ground = widgets.Textarea(placeholder='원자료·실행 출력의 위치 및 비교한 근거',layout=widgets.Layout(width='100%',height='65px'))
            action = widgets.Textarea(placeholder='맞음·수정·보류, 수정 내용 및 재확인 결과',layout=widgets.Layout(width='100%',height='65px'))
            reviews.append((claim,ground,action))
            record_items.append(widgets.VBox([widgets.HTML(f'<b>검토 {i+1} (최소 3건 중 {i+1}번째)</b>'),claim,ground,action]))

        status_check = widgets.SelectMultiple(options=['달성','미달','미측정','미실시','추가 확인 필요'],
                                              value=(),description='수행 상태',layout=widgets.Layout(width='70%',height='115px'))
        approval = widgets.Dropdown(options=['미확인 / 교관 승인 대기','실제 교관 검토 완료 (기록 확인 필요)'],
                                    value='미확인 / 교관 승인 대기', description='교관 상태',layout=widgets.Layout(width='85%'))
        record_notice = widgets.HTML('<span style="color:#666">입력 후 [학습기록 저장]을 누르세요. 미작성·미측정은 그대로 표시합니다.</span>')
        record_dirty = {'value': False}
        def _mark_record_dirty(change):
            record_dirty['value'] = True
            record_notice.value = '<span style="color:#a65">● 학습기록에 저장하지 않은 변경이 있습니다.</span>'
        for field in list(record_widgets.values()) + [v for group in reviews for v in group] + [status_check,approval]:
            field.observe(_mark_record_dirty,names='value')

        # 앞서 작성한 학습기록을 자동으로 불러와 이어서 수정할 수 있습니다.
        previous_audit = RECORD_DIR/'learner_audit.json'
        if previous_audit.is_file():
            try:
                previous = json.loads(previous_audit.read_text(encoding='utf-8'))
                for key, field in record_widgets.items():
                    field.value = str(previous.get(key,'') or '')
                for group, past in zip(reviews, previous.get('reviews',[])):
                    for widget, key in zip(group,('claim','evidence','revision_and_recheck')):
                        widget.value = str(past.get(key,'') or '')
                status_check.value = tuple(x for x in previous.get('status',[]) if x in status_check.options)
                if previous.get('instructor_approval') in approval.options:
                    approval.value = previous['instructor_approval']
                record_dirty['value'] = False
                record_notice.value = '<span style="color:#286">이전 학습기록을 불러왔습니다. 수정 후 다시 저장하세요.</span>'
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">기존 학습기록 읽기 오류: '+str(exc)+'</span>'

        def audit_object():
            return {
                'project': PROJECT_NAME, 'mission': MISSION_CODE, 'notebook_role': NOTEBOOK_ROLE,
                'plan':record_widgets['plan'].value, 'ai_request':record_widgets['ai_request'].value,
                'ai_response':record_widgets['ai_response'].value, 'evidence':record_widgets['evidence'].value,
                'revision':record_widgets['revision'].value, 'decision':record_widgets['decision'].value,
                'reviews':[{'claim':a.value,'evidence':b.value,'revision_and_recheck':c.value} for a,b,c in reviews],
                'status':list(status_check.value),'instructor_approval':approval.value,
                'saved_at':datetime.datetime.now().isoformat(timespec='seconds')
            }

        def save_audit(button=None):
            try:
                record = audit_object()
                (RECORD_DIR/'learner_audit.json').write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding='utf-8')
                (RECORD_DIR/'AI_대화원문.txt').write_text('실제 AI 질문 원문\n'+record['ai_request']+'\n\n실제 AI 답변 원문\n'+record['ai_response'],encoding='utf-8')
                sections = [f'# STR01 미션 {MISSION_CODE:02d} 학습기록']
                for name,key in [('학습 계획·수행 기록','plan'),('실제 출력·자료 대조 근거','evidence'),('검토 및 수정 내용','revision'),('최종 판단','decision')]:
                    sections.append('## '+name+'\n'+(record[key] or '(미작성)'))
                for i,item in enumerate(record['reviews'],1):
                    sections.append(f"## 검토 {i}\n- 검토 대상: {item['claim'] or '(미작성)'}\n- 대조 근거: {item['evidence'] or '(미작성)'}\n- 수정 및 재확인: {item['revision_and_recheck'] or '(미작성)'}")
                sections.append('## 상태\n'+(', '.join(record['status']) or '미선택')+'\n\n교관 상태: '+record['instructor_approval'])
                (RECORD_DIR/'학습기록.md').write_text('\n\n'.join(sections)+'\n',encoding='utf-8')
                record_dirty['value'] = False
                (WORK_ROOT / ZIP_FILENAME).unlink(missing_ok=True)
                completed = sum(all(v.strip() for v in [a.value,b.value,c.value]) for a,b,c in reviews)
                record_notice.value = f'<span style="color:green">✓ 학습기록 저장 완료. 근거가 모두 입력된 검토: {completed}/3건. 저장은 PASS나 교관 승인과 다릅니다.</span>'
                return True
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">학습기록 저장 실패: '+str(exc)+'</span>'
                return False

        save_audit_btn = widgets.Button(description='학습기록 저장',icon='save',button_style='success')
        save_audit_btn.on_click(save_audit)
        record_tab = widgets.VBox(record_items+[widgets.HTML('<b>최종 수행 상태</b> — 해당되는 항목을 선택하세요. 선택만으로 자동 평가되지 않습니다.'),status_check,approval,save_audit_btn,record_notice])

        tabs = widgets.Tab(children=file_tabs+[record_tab])
        for i, path in enumerate(DELIVERABLE_PATHS):
            tabs.set_title(i, f'산출물 {i+1}')
        tabs.set_title(len(file_tabs),'학습기록')

        summary_output = widgets.Output()
        all_btn = widgets.Button(description='작성한 산출물 전체 저장', icon='save',button_style='info')
        zip_btn = widgets.Button(description='저장 파일 ZIP 생성',icon='archive',button_style='warning')
        dl_btn = widgets.Button(description='ZIP 다운로드',icon='download',button_style='')

        def save_all(button):
            ok = 0
            for rel in DELIVERABLE_PATHS:
                if editors[rel]['editor'].value.strip() and save_one(rel):
                    ok += 1
            with summary_output:
                summary_output.clear_output()
                print(f'산출물 {ok}개 저장 완료. 파일별 상태를 확인하세요. 학습기록은 별도 [학습기록 저장] 버튼을 누르세요.')

        def build_zip(button):
            with summary_output:
                summary_output.clear_output()
                if record_dirty['value']:
                    print('❌ 학습기록에 미저장 변경 내용이 있습니다. [학습기록 저장]을 먼저 누르세요.')
                    return
                dirty = [p for p,s in editors.items() if s['dirty']]
                if dirty:
                    print('❌ 미저장 변경 내용이 있습니다. 아래 파일들을 먼저 저장하세요:')
                    for p in dirty: print(' -',p)
                    return
                # 기존에 제공된 파일은 ZIP에 임의로 포함하지 않고 현재 화면에서 불러오거나 저장한 파일만 수집합니다.
                saved = [p for p,s in editors.items() if s['saved'] and safe_path(p).is_file()]
                audit_files = [p for p in RECORD_DIR.rglob('*') if p.is_file()]
                if not saved and not audit_files:
                    print('❌ 저장된 결과물이 없습니다. 산출물 또는 학습기록을 저장하세요.')
                    return
                dest = WORK_ROOT / ZIP_FILENAME
                with zipfile.ZipFile(dest,'w',compression=zipfile.ZIP_DEFLATED) as archive:
                    for rel in saved: archive.write(safe_path(rel), arcname=rel)
                    for f in audit_files: archive.write(f, arcname=str(f.relative_to(WORK_ROOT)))
                print('✅ ZIP 생성:', dest)
                print('산출물:',len(saved),'개 / 학습기록:',len(audit_files),'개')
                print('※ ZIP 생성은 학습 완료 또는 교관 승인을 의미하지 않습니다.')

        def download_zip(button):
            with summary_output:
                summary_output.clear_output()
                target = WORK_ROOT / ZIP_FILENAME
                if record_dirty['value'] or any(s['dirty'] for s in editors.values()):
                    print('미저장 변경이 있습니다. 저장 후 ZIP을 다시 생성해 주세요.')
                    return
                if not target.is_file():
                    print('먼저 [저장 파일 ZIP 생성]을 눌러 주세요.')
                    return
                try:
                    from google.colab import files as colab_files
                    colab_files.download(str(target))
                    print('다운로드를 요청했습니다.')
                except ImportError:
                    start_dir = globals().get('notebook_directory', WORK_ROOT.parent)
                    href = quote(Path(os.path.relpath(target, start_dir)).as_posix())
                    display(HTML(f'<a href="{href}" download>ZIP 다운로드: {target.name}</a>'))

        all_btn.on_click(save_all)
        zip_btn.on_click(build_zip)
        dl_btn.on_click(download_zip)

        print(f'✅ STR01 미션 {MISSION_CODE:02d} 산출물·학습기록 입력 화면')
        display(Markdown('**사용 순서:** ① 각 산출물 탭에서 작성하거나 기존 파일 불러오기 → ② 파일별 저장 → ③ 학습기록 탭에서 실제 AI 원문·근거·검토 3건·최종 상태 입력 및 저장 → ④ ZIP 생성 → ⑤ 다운로드.'))
        display(Markdown('**주의:** 원본 실습의 TODO는 직접 실행해야 합니다. 교관용 참고 출력과 실제 수행 기록은 구분하세요. 입력창은 실제 수행 내용을 기록하기 위한 기능이며, 미작성 결과를 자동으로 완성하지 않습니다.'))
        display(tabs,widgets.HBox([all_btn,zip_btn,dl_btn]),summary_output)
